# Clase 168 — Despliegue en Vertex AI

Desplegar un modelo a **Vertex AI** (GCP managed): subir al **Model Registry**, crear un
**Endpoint** con auto-scaling y hacer predicciones. Alternativas: SageMaker, Azure ML, Modal,
Replicate, HuggingFace Endpoints.

Requiere: `google-cloud-aiplatform` (opcional). Clase de despliegue: el código es correcto pero
no se ejecuta (necesita una cuenta GCP con billing).

## 1. Inicializar el SDK y subir el modelo al Model Registry

In [ ]:
try:
    from google.cloud import aiplatform
    GCP_OK = True
except Exception:
    GCP_OK = False
    print("google-cloud-aiplatform no instalado -> se muestra la API (no se ejecuta)")

PROJECT, REGION, BUCKET = "mi-proyecto", "us-central1", "gs://mi-bucket/fashion"

if GCP_OK:
    aiplatform.init(project=PROJECT, location=REGION, staging_bucket=BUCKET)
    model = aiplatform.Model.upload(
        display_name="fashion-mnist",
        artifact_uri=BUCKET,                                   # SavedModel en GCS
        serving_container_image_uri=(
            "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
        ),
    )
    print("modelo subido:", model.resource_name)
else:
    print("aiplatform.init(project=..., location=...); aiplatform.Model.upload(...)")

## 2. Crear un Endpoint y desplegar con auto-scaling

In [ ]:
if GCP_OK:
    endpoint = model.deploy(
        machine_type="n1-standard-4",
        min_replica_count=1,          # evita cold-start (auto-scale a 0 tarda 30-60 s)
        max_replica_count=3,
        traffic_split={"0": 100},
    )
    print("endpoint desplegado:", endpoint.resource_name)
else:
    print("endpoint = model.deploy(machine_type='n1-standard-4',")
    print("                        min_replica_count=1, max_replica_count=3)")

## 3. Hacer predicciones contra el endpoint

In [ ]:
import numpy as np
np.random.seed(42)
instances = np.random.rand(2, 784).astype("float32").tolist()

if GCP_OK:
    resp = endpoint.predict(instances=instances)
    print("predicciones:", np.array(resp.predictions).shape)
else:
    print("endpoint.predict(instances=instances)  -> resp.predictions")

## 4. Traffic split para A/B testing

Se despliega una segunda versión en el mismo endpoint y se reparte el tráfico. Vertex enruta
por porcentaje, permitiendo comparar modelos en producción.

In [ ]:
if GCP_OK:
    endpoint.deploy(model=model_v2, traffic_split={"0": 80, "1": 20})   # 20% a v2
    print("A/B activo: 80% v1 / 20% v2")
else:
    print("endpoint.deploy(model=model_v2, traffic_split={'0': 80, '1': 20})")
    print("cleanup para no facturar:  endpoint.undeploy_all(); endpoint.delete()")

## 5. Managed vs self-hosted y alternativas

| Opción | Ventaja | Cuándo |
|---|---|---|
| **Vertex AI / SageMaker** | cero ops, IAM, auto-scale | equipos chicos, enterprise |
| **TF Serving + GKE/EKS** | más barato a volumen alto | equipo de ops, tráfico alto |
| **Modal / Replicate** | serverless, pay-per-request | LLMs/difusión, DX rápida |

Cuidado con el costo: un endpoint `n1-standard-4` 24/7 ≈ $100/mes; con GPU T4 ≈ $250/mes.
Auto-scale a 0 entre requests reduce mucho el gasto.

## 6. Batch prediction para grandes volúmenes

Para inferir sobre datasets grandes sin necesidad de tiempo real, un **batch prediction job** es
más barato que un endpoint always-on (no paga réplicas ociosas).

In [ ]:
if GCP_OK:
    batch_job = model.batch_predict(
        job_display_name="fashion-batch",
        gcs_source="gs://mi-bucket/inputs/*.jsonl",
        gcs_destination_prefix="gs://mi-bucket/outputs/",
        machine_type="n1-standard-4",
        starting_replica_count=1,
        max_replica_count=5,
    )
    print("batch job:", batch_job.resource_name)
else:
    print("model.batch_predict(gcs_source='...*.jsonl', gcs_destination_prefix='...',")
    print("                    machine_type='n1-standard-4')  # sin endpoint always-on")

## Ejercicios

1. Subir un SavedModel a GCS y registrarlo en el Model Registry con un container prebuilt.
2. Crear un endpoint (`min_replica_count=1`) y hacer 10 predicciones desde un notebook local.
3. Desplegar una v2 con 20% de tráfico y comparar métricas en los logs.
4. Ejecutar el cleanup (`undeploy_all` + `delete`) y verificar que el costo del experimento < $1.

## Conclusiones

- Vertex AI gestiona el serving: Model Registry → Endpoint → predicciones, sin mantener infra.
- El auto-scaling con `min_replica_count=1` evita el cold-start a costa de un mínimo de gasto.
- El traffic split habilita A/B testing entre versiones en el mismo endpoint.
- Managed conviene a equipos chicos; self-hosted (K8s) gana a volumen alto con equipo de ops.